# Lesson 12.3 - LLMOps Foundations & LLM Lifecycle (architecture & lifecycle template)

This notebook demonstrates a lightweight LLMOps workflow without requiring external model downloads:

- prompt versioning,
- retrieval simulation,
- response generation via mock provider,
- evaluation scorecard and release gating.


## Educational Scope and Production Extension Note
This notebook intentionally uses lightweight simulation/stub components for teaching. Treat outputs as instructional, not production-ready.

For production: replace simulated components with real services/models, add reliability controls, and instrument full observability.

## Objectives

1. Treat prompts and retrieval settings as versioned artifacts.
2. Evaluate quality, groundedness, and cost proxies.
3. Simulate release decisions for LLM workflow variants.

In [1]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from typing import Dict, List
import re

import pandas as pd

## Step 1: Define Toy Knowledge Base and Retrieval

In [2]:
KB = [
    {"id": "doc_1", "text": "Our enterprise plan supports SSO, audit logs, and priority support.", "tags": ["pricing", "enterprise"]},
    {"id": "doc_2", "text": "Starter plan includes up to 5 users and community support.", "tags": ["pricing", "starter"]},
    {"id": "doc_3", "text": "Data is encrypted at rest and in transit with role-based access control.", "tags": ["security", "compliance"]},
]


def simple_retrieve(query: str, top_k: int = 2) -> List[Dict[str, str]]:
    q_words = set(re.findall(r"[a-zA-Z]+", query.lower()))
    scored = []
    for doc in KB:
        doc_words = set(re.findall(r"[a-zA-Z]+", doc["text"].lower()))
        score = len(q_words.intersection(doc_words))
        scored.append((score, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for score, doc in scored[:top_k] if score > 0]

## Step 2: Prompt Registry and Mock LLM

In [3]:
PROMPT_REGISTRY = {
    "v1": "Answer the question using the provided context. Be concise.",
    "v2": "Answer only from context. If uncertain, say 'I don't have enough evidence'. Include cited doc IDs.",
}

@dataclass
class LLMRun:
    run_id: str
    prompt_version: str
    query: str
    retrieved_docs: List[str]
    response: str
    grounded: int
    answer_length: int
    estimated_tokens: int


def mock_llm_answer(prompt: str, query: str, retrieved_docs: List[Dict[str, str]]) -> str:
    if not retrieved_docs:
        return "I don't have enough evidence."
    docs_text = " ".join([d["text"] for d in retrieved_docs])
    doc_ids = ", ".join([d["id"] for d in retrieved_docs])
    if "Include cited doc IDs" in prompt:
        return f"Based on context: {docs_text} [sources: {doc_ids}]"
    return f"Based on context: {docs_text}"

## Step 3: Run Two Prompt Variants

In [4]:
queries = [
    "What security features do you provide?",
    "What does the starter plan include?",
    "Do you support enterprise audit logs?",
]

runs: List[LLMRun] = []

for i, q in enumerate(queries, start=1):
    retrieved = simple_retrieve(q, top_k=2)
    for pv in ["v1", "v2"]:
        response = mock_llm_answer(PROMPT_REGISTRY[pv], q, retrieved)
        grounded = int(bool(retrieved) and "Based on context" in response)
        est_tokens = len(response.split()) + len(q.split())
        runs.append(
            LLMRun(
                run_id=f"run_{i}_{pv}",
                prompt_version=pv,
                query=q,
                retrieved_docs=[d["id"] for d in retrieved],
                response=response,
                grounded=grounded,
                answer_length=len(response),
                estimated_tokens=est_tokens,
            )
        )

runs_df = pd.DataFrame([asdict(r) for r in runs])
runs_df.head()

,run_id,prompt_version,query,retrieved_docs,response,grounded,answer_length,estimated_tokens
0,run_1_v1,v1,What security features do you provide?,[],I don't have enough evidence.,0,29,11
1,run_1_v2,v2,What security features do you provide?,[],I don't have enough evidence.,0,29,11
2,run_2_v1,v1,What does the starter plan include?,"[doc_2, doc_1]",Based on context: Starter plan includes up to ...,1,144,29
3,run_2_v2,v2,What does the starter plan include?,"[doc_2, doc_1]",Based on context: Starter plan includes up to ...,1,168,32
4,run_3_v1,v1,Do you support enterprise audit logs?,"[doc_1, doc_2]",Based on context: Our enterprise plan supports...,1,144,29


## Step 4: Evaluation Scorecard and Release Gate

In [5]:
scorecard = (
    runs_df.groupby("prompt_version")
    .agg(
        grounded_rate=("grounded", "mean"),
        avg_tokens=("estimated_tokens", "mean"),
        avg_answer_length=("answer_length", "mean"),
    )
    .reset_index()
)

# Example: prefer high grounded rate and lower token cost.
scorecard["composite_score"] = scorecard["grounded_rate"] - 0.003 * scorecard["avg_tokens"]
scorecard = scorecard.sort_values("composite_score", ascending=False)
scorecard

,prompt_version,grounded_rate,avg_tokens,avg_answer_length,composite_score
0,v1,0.666667,23.0,105.666667,0.597667
1,v2,0.666667,25.0,121.666667,0.591667


In [6]:
release_candidate = scorecard.iloc[0]["prompt_version"]
release_reason = {
    "selected_version": release_candidate,
    "reason": "Highest composite score with groundedness and token efficiency balance.",
}
release_reason

{'selected_version': 'v1',
 'reason': 'Highest composite score with groundedness and token efficiency balance.'}

## Production Extension Checklist
- Replace simulated/stub components with production services or managed endpoints.
- Add structured logging, tracing IDs, and latency/error dashboards.
- Add input/output validation and policy/safety guardrails.
- Add retry/timeouts, rate limiting, and fallback behavior.
- Add offline/online evaluation gates before release.
- Add secrets management and environment-specific configuration.
- Add CI checks and smoke tests for critical paths.

## Production Case Studies & Exceptions

### Case 1: Support copilot with stale retrieval index
- Prompt quality looked stable, but answers degraded due to outdated index.
- Added index freshness checks and ingestion SLOs.

### Case 2: Cost spike from verbose responses
- Prompt update increased answer verbosity and token spend.
- Added response-length constraints and prompt regression tests.

### Case 3 (Exception): Deterministic workflow required
- In high-regulation workflows, team used LLM for draft generation only; final decision stayed rule-based.

## Interview Questions & Answers

1. **Q: Why version prompts in LLMOps?**
   **A:** Prompt changes can materially alter behavior, quality, and cost.

2. **Q: What is groundedness?**
   **A:** Degree to which output is supported by provided trusted context.

3. **Q: How do you evaluate prompt versions?**
   **A:** Compare quality, safety, latency, and cost on a stable evaluation set.

4. **Q: What is a prompt regression test?**
   **A:** A repeatable suite ensuring prompt updates don't break critical behaviors.

5. **Q: How does retrieval quality affect LLM outputs?**
   **A:** Poor retrieval often causes confident but unsupported answers.

6. **Q: Why monitor token usage?**
   **A:** Token volume is a major runtime cost driver.

7. **Q: When should LLM output be constrained?**
   **A:** When compliance or deterministic formatting requirements are strict.

8. **Q: What is LLM release gating?**
   **A:** Promotion based on quality/safety/cost thresholds.

9. **Q: How do you handle no-evidence cases?**
   **A:** Use abstention logic and explicit uncertainty responses.

10. **Q: Why combine automated and human evals?**
   **A:** Automated checks scale; human review captures nuanced quality/risk.
